# PySpark NLP and Chatbot Practice in Google Colab

This notebook is designed as a **replacement for the Tweepy/Twitter notebook** when API tokens or subscriptions are unavailable.

It keeps the same **instructional style**:
- setup
- load text data
- run sentiment analysis
- build NLP features with PySpark
- create a simple chatbot workflow

## What this notebook teaches
1. How to run **PySpark in Colab**
2. How to create a small **sample text dataset**
3. How to perform **sentiment analysis** without Twitter API keys
4. How to use **PySpark NLP-style preprocessing**
5. How to build a simple **retrieval chatbot**
6. Optional: how to use a **small local transformer model** without paid API access


## Step 1. Install dependencies
This notebook uses:
- PySpark
- VADER for rule-based sentiment
- scikit-learn for a lightweight retrieval chatbot
- transformers for optional local text generation


In [ ]:
!pip -q install pyspark vaderSentiment scikit-learn transformers sentencepiece


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 3.5 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2. Start a Spark session


In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("PySpark_NLP_Chatbot_Colab") \
    .master("local[*]") \
    .getOrCreate()

print("Spark version:", spark.version)


Spark version: 4.0.2


## Step 3. Create sample text data

Instead of pulling tweets from Twitter, we create a small dataset of messages that look like:
- customer feedback
- chatbot prompts
- support requests
- positive and negative sentiment examples

This makes the notebook fully runnable in Colab.


In [ ]:
sample_data = [
    (1, "customer_review", "I absolutely love this laptop. It is fast, light, and reliable."),
    (2, "customer_review", "The battery life is terrible and the screen freezes often."),
    (3, "support_chat", "My internet keeps disconnecting every hour. Can you help?"),
    (4, "support_chat", "Thank you, the issue is fixed now and everything works great."),
    (5, "chatbot_prompt", "How do I reset my password for the student portal?"),
    (6, "chatbot_prompt", "Can I get a refund for a damaged order?"),
    (7, "feedback", "The class was engaging, practical, and easy to follow."),
    (8, "feedback", "The lecture was confusing and much too rushed."),
    (9, "support_chat", "I am frustrated because the package arrived late again."),
    (10, "support_chat", "The new update is amazing and much easier to use."),
    (11, "faq", "Password reset instructions: click Forgot Password and follow the email link."),
    (12, "faq", "Refund policy: refunds are allowed within 30 days with proof of purchase."),
    (13, "faq", "Shipping delays may occur during holidays or severe weather."),
    (14, "faq", "You can contact support by email, chat, or phone."),
]

df = spark.createDataFrame(sample_data, ["id", "category", "text"])
df.show(truncate=False)


+---+---------------+-----------------------------------------------------------------------------+
|id |category       |text                                                                         |
+---+---------------+-----------------------------------------------------------------------------+
|1  |customer_review|I absolutely love this laptop. It is fast, light, and reliable.              |
|2  |customer_review|The battery life is terrible and the screen freezes often.                   |
|3  |support_chat   |My internet keeps disconnecting every hour. Can you help?                    |
|4  |support_chat   |Thank you, the issue is fixed now and everything works great.                |
|5  |chatbot_prompt |How do I reset my password for the student portal?                           |
|6  |chatbot_prompt |Can I get a refund for a damaged order?                                      |
|7  |feedback       |The class was engaging, practical, and easy to follow.                       |


## Step 4. Inspect the data


In [ ]:
print("Row count:", df.count())
print("Columns:", df.columns)
df.printSchema()


Row count: 14
Columns: ['id', 'category', 'text']
root
 |-- id: long (nullable = true)
 |-- category: string (nullable = true)
 |-- text: string (nullable = true)



## Step 5. Rule-based sentiment analysis with VADER

This replaces the Twitter sentiment part with a local sentiment analyzer that runs without keys.


In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType, StringType

analyzer = SentimentIntensityAnalyzer()

def vader_compound(text):
    return float(analyzer.polarity_scores(text)["compound"])

def vader_label(text):
    score = analyzer.polarity_scores(text)["compound"]
    if score >= 0.05:
        return "positive"
    elif score <= -0.05:
        return "negative"
    return "neutral"

compound_udf = udf(vader_compound, DoubleType())
label_udf = udf(vader_label, StringType())

sentiment_df = df.withColumn("compound_score", compound_udf("text")) \
                 .withColumn("sentiment_label", label_udf("text"))

sentiment_df.select("id", "category", "text", "compound_score", "sentiment_label").show(truncate=False)


+---+---------------+-----------------------------------------------------------------------------+--------------+---------------+
|id |category       |text                                                                         |compound_score|sentiment_label|
+---+---------------+-----------------------------------------------------------------------------+--------------+---------------+
|1  |customer_review|I absolutely love this laptop. It is fast, light, and reliable.              |0.6697        |positive       |
|2  |customer_review|The battery life is terrible and the screen freezes often.                   |-0.4939       |negative       |
|3  |support_chat   |My internet keeps disconnecting every hour. Can you help?                    |0.4019        |positive       |
|4  |support_chat   |Thank you, the issue is fixed now and everything works great.                |0.765         |positive       |
|5  |chatbot_prompt |How do I reset my password for the student portal?            

### Explanation
- `compound_score` ranges from -1 to 1
- positive values indicate more positive sentiment
- negative values indicate more negative sentiment
- this is a good instructional substitute for tweet sentiment analysis


## Step 6. Aggregate sentiment results in PySpark


In [ ]:
from pyspark.sql.functions import count, avg

sentiment_df.groupBy("sentiment_label").agg(
    count("*").alias("num_rows"),
    avg("compound_score").alias("avg_score")
).show()


+---------------+--------+------------------+
|sentiment_label|num_rows|         avg_score|
+---------------+--------+------------------+
|       positive|       6|0.6086833333333334|
|        neutral|       4|               0.0|
|       negative|       4|         -0.421825|
+---------------+--------+------------------+



## Step 7. Basic NLP preprocessing with PySpark ML

We now build a standard NLP feature pipeline using:
- RegexTokenizer
- StopWordsRemover
- CountVectorizer
- IDF

This is useful for teaching how raw text becomes machine-learning features.


In [ ]:
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover, CountVectorizer, IDF
from pyspark.ml import Pipeline

tokenizer = RegexTokenizer(
    inputCol="text",
    outputCol="tokens",
    pattern="\\W+"
)

remover = StopWordsRemover(
    inputCol="tokens",
    outputCol="filtered_tokens"
)

vectorizer = CountVectorizer(
    inputCol="filtered_tokens",
    outputCol="raw_features",
    vocabSize=1000,
    minDF=1
)

idf = IDF(
    inputCol="raw_features",
    outputCol="tfidf_features"
)

pipeline = Pipeline(stages=[tokenizer, remover, vectorizer, idf])
pipeline_model = pipeline.fit(df)
nlp_df = pipeline_model.transform(df)

nlp_df.select("id", "text", "tokens", "filtered_tokens").show(truncate=False)


+---+-----------------------------------------------------------------------------+---------------------------------------------------------------------------------------+-----------------------------------------------------------------------------+
|id |text                                                                         |tokens                                                                                 |filtered_tokens                                                              |
+---+-----------------------------------------------------------------------------+---------------------------------------------------------------------------------------+-----------------------------------------------------------------------------+
|1  |I absolutely love this laptop. It is fast, light, and reliable.              |[i, absolutely, love, this, laptop, it, is, fast, light, and, reliable]                |[absolutely, love, laptop, fast, light, reliable]                            |


## Step 8. View TF-IDF feature vectors


In [ ]:
nlp_df.select("id", "category", "tfidf_features").show(truncate=False)


+---+---------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|id |category       |tfidf_features                                                                                                                                                                                              |
+---+---------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|1  |customer_review|(74,[22,24,36,44,52,71],[2.0149030205422647,2.0149030205422647,2.0149030205422647,2.0149030205422647,2.0149030205422647,2.0149030205422647])                                                                |
|2  |customer_review|(74,[11,12,19,34,46,67],[2.0149030205422647,2.0149030205422647,2.014903

### Explanation
TF-IDF helps identify important words in each sentence:
- common words get lower weight
- more informative words get higher weight

This is a core NLP concept and also useful for retrieval-based chatbots.


## Step 9. Build a simple retrieval chatbot

Since you asked for something similar to a chatbot that works in Colab, this section creates a small FAQ chatbot.

How it works:
1. store FAQ entries
2. vectorize them with TF-IDF
3. compare the user's question to FAQ entries
4. return the most similar answer


In [ ]:
faq_pairs = [
    ("How do I reset my password?", "To reset your password, click 'Forgot Password' and follow the email instructions."),
    ("Can I get a refund?", "Refunds are allowed within 30 days with proof of purchase."),
    ("Why is my order late?", "Shipping delays may occur during holidays, weather events, or carrier issues."),
    ("How can I contact support?", "You can contact support by email, live chat, or phone."),
    ("What if my item is damaged?", "Please contact support and provide a photo so we can replace or refund the item."),
]

questions = [q for q, a in faq_pairs]
answers = [a for q, a in faq_pairs]

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer()
faq_matrix = vectorizer.fit_transform(questions)

def chatbot_reply(user_question):
    user_vec = vectorizer.transform([user_question])
    sims = cosine_similarity(user_vec, faq_matrix)[0]
    best_idx = sims.argmax()
    return {
        "user_question": user_question,
        "best_match_question": questions[best_idx],
        "answer": answers[best_idx],
        "similarity_score": float(sims[best_idx])
    }

test_questions = [
    "I forgot my password. What should I do?",
    "Can I return something for money back?",
    "How do I reach customer service?",
    "My package is delayed."
]

for q in test_questions:
    result = chatbot_reply(q)
    print("=" * 80)
    print("User question :", result["user_question"])
    print("Matched FAQ   :", result["best_match_question"])
    print("Answer        :", result["answer"])
    print("Similarity    :", round(result["similarity_score"], 4))


User question : I forgot my password. What should I do?
Matched FAQ   : How do I reset my password?
Answer        : To reset your password, click 'Forgot Password' and follow the email instructions.
Similarity    : 0.6512
User question : Can I return something for money back?
Matched FAQ   : Can I get a refund?
Answer        : Refunds are allowed within 30 days with proof of purchase.
Similarity    : 0.4955
User question : How do I reach customer service?
Matched FAQ   : How do I reset my password?
Answer        : To reset your password, click 'Forgot Password' and follow the email instructions.
Similarity    : 0.6346
User question : My package is delayed.
Matched FAQ   : Why is my order late?
Answer        : Shipping delays may occur during holidays, weather events, or carrier issues.
Similarity    : 0.5179


## Step 10. Turn the retrieval chatbot into a mini interactive cell


In [ ]:
user_question = "How can I change my password?"
result = chatbot_reply(user_question)

print("Question:", result["user_question"])
print("Best match:", result["best_match_question"])
print("Bot answer:", result["answer"])
print("Similarity:", round(result["similarity_score"], 4))


Question: How can I change my password?
Best match: How do I reset my password?
Bot answer: To reset your password, click 'Forgot Password' and follow the email instructions.
Similarity: 0.6252


## Step 11. Optional local transformer demo

This section is optional.  
It uses a **small local Hugging Face model** in Colab instead of a paid API.

This is useful if you want to show a more modern LLM-style workflow without relying on subscriptions.


In [ ]:
from transformers import pipeline

sentiment_pipe = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

examples = [
    "This notebook is incredibly useful and easy to understand.",
    "The instructions are confusing and the results are disappointing."
]

for text in examples:
    print(text)
    print(sentiment_pipe(text))
    print("-" * 80)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

This notebook is incredibly useful and easy to understand.
[{'label': 'POSITIVE', 'score': 0.9995031356811523}]
--------------------------------------------------------------------------------
The instructions are confusing and the results are disappointing.
[{'label': 'NEGATIVE', 'score': 0.9997966885566711}]
--------------------------------------------------------------------------------


## Step 12. Optional text generation demo

This is a tiny local generation example.  
It is not as strong as a hosted LLM, but it demonstrates the concept in Colab.


In [ ]:
generator = pipeline("text-generation", model="sshleifer/tiny-gpt2")

prompt = "Customer: My internet keeps dropping.\nSupport:"
result = generator(prompt, max_new_tokens=30, do_sample=True, temperature=0.8)

print(result[0]["generated_text"])


config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.51M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/29 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: sshleifer/tiny-gpt2
Key                                   | Status     |  | 
--------------------------------------+------------+--+-
transformer.h.{0, 1}.attn.masked_bias | UNEXPECTED |  | 
transformer.h.{0, 1}.attn.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Customer: My internet keeps dropping.
Support: intermittent Habit Brewhibit confir substJD Prob TA confir confir Observ vendors reviewing reborn Prob stairsJDoho hauled ONE TA MoneyRocketmediatelyiken Participationhibit vendorsSher


## Step 13. Sample questions and answers

Use the outputs above to answer the following questions.

### Q1. Why does this notebook not need Tweepy?
**Answer:** Because it uses local sample text data and local NLP models instead of live Twitter API calls.

### Q2. What does VADER measure?
**Answer:** It measures sentiment polarity and returns a score showing how positive or negative text is.

### Q3. Why use TF-IDF?
**Answer:** TF-IDF converts text into numeric features and highlights informative words.

### Q4. How does the chatbot work?
**Answer:** It compares the user's question with stored FAQ questions and returns the most similar answer.

### Q5. What is the difference between the retrieval chatbot and the local transformer demo?
**Answer:** The retrieval chatbot selects from known answers, while the transformer demo generates new text.
